In [2]:
import numpy as np
import os
import cv2
from imutils import paths
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from IPython.display import Image

# 1. Извлечение признаков (гистограммы)
def extract_histogram(image, bins=(8, 8, 8)):
    hist = cv2.calcHist([image], [0, 1, 2], None, bins, [0, 256, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

# Загрузка путей
imagePaths = sorted(list(paths.list_images('../Practice6/train')))
data = []
labels = []

for imagePath in imagePaths:
    image = cv2.imread(imagePath, 1)
    # Извлекаем метку из имени файла (предполагаем, что имя вида cat.xxx.jpg или dog.xxx.jpg)
    label = imagePath.split(os.path.sep)[-1].split(".")[0]
    hist = extract_histogram(image)
    data.append(hist)
    labels.append(label)

data = np.array(data)
Y = np.array([1 if x == 'cat' else 0 for x in labels])

# 2. Базовые алгоритмы
# 2.1 SVM с мягким зазором
svm = LinearSVC(C=1.74, random_state=220)

# 2.2 Бэггинг деревьев решений
tree = DecisionTreeClassifier(
    criterion='entropy',
    min_samples_leaf=10,
    max_leaf_nodes=20,
    random_state=220
)
bagging = BaggingClassifier(
    estimator=tree,
    n_estimators=19,
    random_state=220
)

# 2.3 Случайный лес
rf = RandomForestClassifier(
    n_estimators=19,
    criterion='entropy',
    min_samples_leaf=10,
    max_leaf_nodes=20,
    random_state=220
)

# 3. Метаалгоритм (логистическая регрессия)
meta_lr = LogisticRegression(
    solver='lbfgs',
    random_state=220
)

# Оценка точности метаалгоритма на кросс-валидации (cv=2)
# (Оцениваем на всем наборе data, Y, т.к. метаалгоритм обучается на предсказаниях базовых)
# Но корректнее оценить именно пайплайн стэкинга. 
# Сначала оценим саму логистическую регрессию на исходных данных для сравнения (не совсем то, что просят, но для понимания)
# А для стэкинга используем StackingClassifier

# 4. Модель стэкинга
stacking = StackingClassifier(
    estimators=[
        ('svm', svm),
        ('bagging', bagging),
        ('rf', rf)
    ],
    final_estimator=meta_lr,
    cv=2
)

# Оценка стэкинга с помощью кросс-валидации (cv=2)
stacking_accuracy = cross_val_score(stacking, data, Y, cv=2, scoring='accuracy').mean()
print(f"Accuracy стэкинга (cv=2): {stacking_accuracy:.2f}")

# Обучаем стэкинг на всех данных, чтобы потом делать предсказания
stacking.fit(data, Y)

# 5. Предсказание для новых изображений
# Предполагаем, что тестовые изображения лежат в папке '../Practice6/test'
test_imagePaths = sorted(list(paths.list_images('../Practice6/test')))

# Создадим словарь для удобства: имя файла -> изображение (для проверки)
test_images_dict = {}

for imagePath in test_imagePaths:
    filename = imagePath.split(os.path.sep)[-1]
    image = cv2.imread(imagePath, 1)
    hist = extract_histogram(image)
    
    # Вероятность класса 1 (cat) с помощью стэкинга
    # StackingClassifier имеет метод predict_proba, если final_estimator поддерживает его (LogisticRegression поддерживает)
    prob = stacking.predict_proba([hist])[0][1]
    print(f"{filename}: {prob:.3f}")

Accuracy стэкинга (cv=2): 0.65
cat.1000.jpg: 0.562
cat.1001.jpg: 0.285
cat.1002.jpg: 0.676
cat.1003.jpg: 0.690
cat.1004.jpg: 0.508
cat.1005.jpg: 0.218
cat.1006.jpg: 0.699
cat.1007.jpg: 0.634
cat.1008.jpg: 0.398
cat.1009.jpg: 0.474
cat.1010.jpg: 0.459
cat.1011.jpg: 0.696
cat.1012.jpg: 0.257
cat.1013.jpg: 0.309
cat.1014.jpg: 0.739
cat.1015.jpg: 0.679
cat.1016.jpg: 0.734
cat.1017.jpg: 0.743
cat.1018.jpg: 0.586
cat.1019.jpg: 0.547
cat.1020.jpg: 0.394
cat.1021.jpg: 0.677
cat.1022.jpg: 0.451
cat.1023.jpg: 0.577
cat.1024.jpg: 0.625
cat.1025.jpg: 0.686
cat.1026.jpg: 0.441
cat.1027.jpg: 0.487
cat.1028.jpg: 0.485
cat.1029.jpg: 0.634
cat.1030.jpg: 0.281
cat.1031.jpg: 0.663
cat.1032.jpg: 0.699
cat.1033.jpg: 0.711
cat.1034.jpg: 0.600
cat.1035.jpg: 0.763
cat.1036.jpg: 0.748
cat.1037.jpg: 0.702
cat.1038.jpg: 0.520
cat.1039.jpg: 0.743
cat.1040.jpg: 0.673
cat.1041.jpg: 0.275
cat.1042.jpg: 0.343
cat.1043.jpg: 0.384
cat.1044.jpg: 0.527
cat.1045.jpg: 0.739
cat.1046.jpg: 0.478
cat.1047.jpg: 0.777
cat.1048.